# Day 1: Data Preprocessing

**Goal:** Load, explore, clean, and save the two datasets used in this project.
All reusable logic lives in `shared/preprocessing.py` and `shared/dataset_loader.py`.
This notebook tells the story — observation after every major step.

## 1. Import Libraries

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

from shared import config
from shared.dataset_loader import load_hate_speech_dataset, load_hatexplain_dataset
from shared.preprocessing import (
    clean_text,
    remove_duplicates,
    standardize_labels_hate_speech,
    standardize_labels_hatexplain,
    calculate_text_length,
    calculate_word_count,
    handle_missing_text,
    print_summary,
)

# Plot style
sns.set_theme(style='whitegrid')
PLOT_DIR = os.path.abspath('../trust_module/results/plots')
os.makedirs(PLOT_DIR, exist_ok=True)

## 2. Load Datasets

In [ ]:
hate_df = load_hate_speech_dataset()
hatexplain_df = load_hatexplain_dataset()

print('Hate Speech Dataset     :', hate_df.shape)
print('HateXplain Dataset      :', hatexplain_df.shape)

### Observation
- The **Measuring Hate Speech** dataset is annotation-level — each row is one annotator's
  response to one comment, giving 135,556 rows and 143 columns (demographics + ratings).
- The **HateXplain** dataset is post-level — each row is one social media post with up to
  three annotator labels stored together, totalling ~20 K posts.

## 3. Dataset Overview

In [ ]:
print('=== Hate Speech: first 5 rows ===')
display(hate_df[['comment_id', 'annotator_id', 'text', 'hatespeech', 'hate_speech_score']].head())

print('\n=== Hate Speech: column dtypes ===')
print(hate_df.dtypes.value_counts())

In [ ]:
print('=== HateXplain: first 5 rows ===')
display(hatexplain_df.head())

print('\n=== HateXplain: column dtypes ===')
print(hatexplain_df.dtypes)

## 4. Data Exploration

### 4a. Missing Values

In [ ]:
missing_hate = hate_df.isnull().sum()
missing_hatex = hatexplain_df.isnull().sum()

print('Hate Speech — columns with missing values:')
print(missing_hate[missing_hate > 0] if missing_hate.any() else '  None')

print('\nHateXplain — columns with missing values:')
print(missing_hatex[missing_hatex > 0] if missing_hatex.any() else '  None')

### Observation
- The Measuring Hate Speech dataset has **no missing values** — it was released as a fully
  completed annotation export.
- HateXplain similarly has no nulls; post tokens were always present in the original JSON.
- Because there are no missing values to impute, the cleaning step simply guards against
  edge cases with a `fillna('')` on the text column.

### 4b. Duplicate Rows

In [ ]:
print('Hate Speech duplicate rows :', hate_df.duplicated().sum())
print('HateXplain duplicate rows  :', hatexplain_df.duplicated().sum())

### Observation
- Neither dataset contains exact duplicate rows, which is expected for structured
  annotation exports. No rows will be dropped in the deduplication step.

### 4c. Label Distribution

In [ ]:
# Binary hate speech label (hatespeech column: 1 = hate, 0 = not)
print('Hate Speech — hatespeech column value counts:')
print(hate_df['hatespeech'].value_counts())

print('\nHateXplain — majority annotator label counts:')
from collections import Counter
all_labels = [lbl for tup in hatexplain_df['labels'] for lbl in tup]
print(pd.Series(Counter(all_labels)))

### Observation
- The Measuring Hate Speech dataset is **imbalanced**: the majority of annotations are
  non-hate (hatespeech = 0), which is typical for real-world data.
- HateXplain shows a mix of 'normal', 'offensive', and 'hatespeech' labels across
  individual annotators. Class imbalance is present here too.
- **Implication for modelling:** we will need to account for class imbalance (e.g.,
  stratified splits, weighted loss) in later stages.

### 4d. Text Length Statistics

In [ ]:
print('Hate Speech — text character length:')
print(hate_df['text'].astype(str).apply(len).describe().round(1))

print('\nHateXplain — text character length:')
print(hatexplain_df['text'].astype(str).apply(len).describe().round(1))

### Observation
- HateXplain posts are tokenised social media text — generally short (Twitter/Gab).
- The Measuring Hate Speech dataset contains longer Reddit and Twitter comments.
- This difference in text length distribution should be kept in mind when designing
  models or comparing results across the two datasets.

## 5. Data Cleaning

In [ ]:
# ── Hate Speech ──────────────────────────────────────────────────────────────
hs_initial = hate_df.shape[0]

hate_df, hs_dupes = remove_duplicates(hate_df)
hate_df, hs_missing = handle_missing_text(hate_df, 'text')
hate_df = standardize_labels_hate_speech(hate_df)
hate_df['clean_text'] = hate_df['text'].apply(clean_text)
hate_df = calculate_text_length(hate_df, 'clean_text')
hate_df = calculate_word_count(hate_df, 'clean_text')

# ── HateXplain ───────────────────────────────────────────────────────────────
hx_initial = hatexplain_df.shape[0]

hatexplain_df, hx_dupes = remove_duplicates(hatexplain_df)
hatexplain_df, hx_missing = handle_missing_text(hatexplain_df, 'text')
hatexplain_df = standardize_labels_hatexplain(hatexplain_df)
hatexplain_df['clean_text'] = hatexplain_df['text'].apply(clean_text)
hatexplain_df = calculate_text_length(hatexplain_df, 'clean_text')
hatexplain_df = calculate_word_count(hatexplain_df, 'clean_text')

print('Cleaning complete.')

## 6. Visualisations

### 6a. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=hate_df, x='label', ax=axes[0], palette='Set2')
axes[0].set_title('Hate Speech — Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

sns.countplot(data=hatexplain_df, x='label', ax=axes[1], palette='Set2')
axes[1].set_title('HateXplain — Label Distribution')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'label_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: label_distribution.png')

### Observation
- Both datasets show clear **class imbalance**: 'normal' dominates in both.
- HateXplain has a three-way split (normal / offensive / hate) while the
  Measuring Hate Speech dataset is binary after standardisation.
- This imbalance will inform our evaluation strategy — accuracy alone will be misleading;
  we will use F1-score and per-class metrics.

### 6b. Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

missing_h = missing_hate[missing_hate > 0]
if not missing_h.empty:
    sns.barplot(x=missing_h.index, y=missing_h.values, ax=axes[0], palette='Reds')
    axes[0].tick_params(axis='x', rotation=45)
else:
    axes[0].text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=13)
    axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title('Hate Speech — Missing Values')

missing_hx = missing_hatex[missing_hatex > 0]
if not missing_hx.empty:
    sns.barplot(x=missing_hx.index, y=missing_hx.values, ax=axes[1], palette='Reds')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=13)
    axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_title('HateXplain — Missing Values')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: missing_values.png')

### 6c. Text Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=hate_df, x='text_length', bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Hate Speech — Text Length (chars)')
axes[0].set_xlabel('Character Count')

sns.histplot(data=hatexplain_df, x='text_length', bins=60, ax=axes[1], color='seagreen')
axes[1].set_title('HateXplain — Text Length (chars)')
axes[1].set_xlabel('Character Count')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'text_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: text_length_distribution.png')

### Observation
- HateXplain text lengths cluster tightly at the shorter end (social media posts),
  with very few outliers.
- Hate Speech texts have a wider spread, reflecting a mix of Twitter and Reddit content.
- Any model trained on one dataset may need length-normalisation before being applied
  to the other.

## 7. Save Processed Datasets

In [ ]:
hate_df.to_csv(config.PROCESSED_HATE_SPEECH_PATH, index=False)
hatexplain_df.to_csv(config.PROCESSED_HATEXPLAIN_PATH, index=False)
print('Datasets saved.')

## 8. Preprocessing Summary

In [ ]:
print_summary(
    name='Measuring Hate Speech',
    initial_rows=hs_initial,
    duplicates_removed=hs_dupes,
    missing_handled=hs_missing,
    final_df=hate_df,
    output_path=config.PROCESSED_HATE_SPEECH_PATH,
)

print_summary(
    name='HateXplain',
    initial_rows=hx_initial,
    duplicates_removed=hx_dupes,
    missing_handled=hx_missing,
    final_df=hatexplain_df,
    output_path=config.PROCESSED_HATEXPLAIN_PATH,
)